<a href="https://colab.research.google.com/github/bhanu613/alias-aware-technical-skill-extraction/blob/main/notebooks/03%20Gold%20Annotation%20Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gold Annotation Workflow and Final Validation

## Purpose

This notebook documents the independent human-annotation workflow used to
create the frozen gold-label dataset for the held-out 100-document evaluation
corpus.

The standard public workflow is read-only. It loads and validates the released
final gold annotations, creates a separate blank practice template in the
runtime output folder, and does not modify the official gold dataset.

## Why Gold Labels Were Needed

The study compares two frozen lexicon-based extraction systems:

- System A matches canonical technical-skill forms only.
- System B matches the same canonical forms plus approved aliases.

A human-created gold standard was needed to evaluate both systems against the
same independent reference labels. Gold labels were assigned from the complete
job-description text, not from System A predictions, System B predictions,
regex matches, safety-test results, or performance metrics.

## Annotation Task

The task is document-level technical-skill mention extraction.

For each complete `Long Description`, the annotator assigned every in-scope
canonical technical concept that was explicitly mentioned or unambiguously
represented anywhere in the full document.

The task uses a bounded 20-label inventory. It does not attempt to identify
every possible technical skill in a job posting, and it is not limited to
applicant-requirement sections.

## Annotation Protocol

The following rules governed independent gold annotation:

- Read the complete job-description text before assigning labels.
- Assign labels only from the frozen 20-label canonical inventory.
- Assign a canonical label when evidence is explicit or unambiguous.
- Treat clear alternate expressions as the relevant canonical concept.
- Do not infer a skill from the job title alone.
- Use `none` when no in-scope canonical technical concept is explicitly present.
- Record genuine uncertainty separately rather than silently guessing.
- Do not inspect system predictions, aliases, regex matches, or metrics during
  primary annotation.

The first 10 documents formed a deterministic pilot. The pilot was used to
confirm that the annotation instructions, 20-label inventory, template, and
human annotation interface were usable. The same fixed protocol was then used
for all 100 held-out documents.

The final released gold file is the frozen outcome of that process. This
notebook validates the released artifact; it does not claim to regenerate human
judgements automatically.

In [14]:
from pathlib import Path

import json
import os
import subprocess
import sys

import pandas as pd


repositoryUrl = (
    "https://github.com/bhanu613/"
    "alias-aware-technical-skill-extraction.git"
)


repositoryFolder = Path(
    "/content/alias-aware-technical-skill-extraction"
)


if not repositoryFolder.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repositoryUrl,
            str(repositoryFolder)
        ],
        check=True
    )


os.chdir(
    repositoryFolder
)


requirementsPath = (
    repositoryFolder
    / "requirements.txt"
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirementsPath)
    ],
    check=True
)


dataFolder = repositoryFolder / "data"

configFolder = repositoryFolder / "config"

runtimeFolder = Path(
    "/content/goldAnnotationOutputs"
)


runtimeFolder.mkdir(
    parents=True,
    exist_ok=True
)


evaluationPath = (
    dataFolder
    / "evaluation100.csv"
)


goldPath = (
    dataFolder
    / "goldAnnotationFinal.csv"
)


lexiconPath = (
    configFolder
    / "LexiconList20_final.json"
)


assert evaluationPath.exists(), (
    "The frozen evaluation100.csv file was not found."
)


assert goldPath.exists(), (
    "The frozen goldAnnotationFinal.csv file was not found."
)


assert lexiconPath.exists(), (
    "The frozen LexiconList20_final.json file was not found."
)


evaluationData = pd.read_csv(
    evaluationPath
)


goldAnnotations = pd.read_csv(
    goldPath
)


with lexiconPath.open(
    "r",
    encoding="utf-8"
) as file:

    finalLexicon = json.load(
        file
    )


canonicalLabels = sorted(
    finalLexicon[
        "skills"
    ].keys()
)


assert len(canonicalLabels) == 20, (
    "Expected exactly 20 frozen canonical labels."
)


print(
    "Frozen annotation artifacts loaded successfully."
)


print(
    f"Evaluation documents: {len(evaluationData)}"
)


print(
    f"Final gold-annotation rows: {len(goldAnnotations)}"
)


print(
    f"Canonical annotation labels: {len(canonicalLabels)}"
)

Frozen annotation artifacts loaded successfully.
Evaluation documents: 100
Final gold-annotation rows: 100
Canonical annotation labels: 20


## Evaluation Alignment and Blank Template

The annotation template is constructed from the fixed held-out evaluation
documents. It contains document metadata and empty annotation fields only.

The template is aligned with the final frozen gold file by annotation order and
document ID. The template itself contains no labels, predictions, or metric
information.

In [15]:
requiredEvaluationColumns = {
    "Annotation order",
    "id",
    "Position",
    "Company Name",
    "Primary Keyword",
    "charc_len",
    "Long Description"
}


requiredGoldColumns = {
    "Annotation order",
    "id",
    "Position",
    "charc_len",
    "Long Description",
    "Gold skills",
    "Uncertain",
    "Annotation status"
}


assert requiredEvaluationColumns.issubset(
    evaluationData.columns
), (
    "evaluation100.csv is missing one or more required columns."
)


assert requiredGoldColumns.issubset(
    goldAnnotations.columns
), (
    "goldAnnotationFinal.csv is missing one or more required columns."
)


assert len(evaluationData) == 100, (
    "Expected exactly 100 held-out evaluation documents."
)


assert len(goldAnnotations) == 100, (
    "Expected exactly 100 frozen gold-annotation rows."
)


assert evaluationData[
    "Annotation order"
].tolist() == list(
    range(
        1,
        101
    )
), (
    "Evaluation annotation order must run from 1 through 100."
)


assert goldAnnotations[
    "Annotation order"
].tolist() == list(
    range(
        1,
        101
    )
), (
    "Gold annotation order must run from 1 through 100."
)


assert evaluationData[
    "id"
].tolist() == goldAnnotations[
    "id"
].tolist(), (
    "Evaluation and gold document IDs are not aligned."
)


assert evaluationData[
    "Annotation order"
].tolist() == goldAnnotations[
    "Annotation order"
].tolist(), (
    "Evaluation and gold annotation orders are not aligned."
)


assert evaluationData[
    "Long Description"
].tolist() == goldAnnotations[
    "Long Description"
].tolist(), (
    "Evaluation and gold document text are not aligned."
)


goldAnnotationTemplate = evaluationData[
    [
        "Annotation order",
        "id",
        "Position",
        "charc_len",
        "Long Description"
    ]
].copy()


goldAnnotationTemplate[
    "Gold skills"
] = ""


goldAnnotationTemplate[
    "Uncertain"
] = ""


goldAnnotationTemplate[
    "Annotator note"
] = ""


goldAnnotationTemplate[
    "Annotation status"
] = "not started"


goldAnnotationTemplate.to_csv(
    runtimeFolder
    / "GoldAnnotationTemplate.csv",
    index=False
)


templatePreview = goldAnnotationTemplate[
    [
        "Annotation order",
        "Position",
        "charc_len",
        "Gold skills",
        "Uncertain",
        "Annotation status"
    ]
].head(
    10
)


display(
    templatePreview
)


print(
    "Evaluation and frozen gold alignment passed."
)


print(
    "A separate blank annotation template was created "
    "in runtime outputs."
)

,Annotation order,Position,charc_len,Gold skills,Uncertain,Annotation status
0,1,"Backend developer (Python, Golang or C#) Short...",3733,,,not started
1,2,Data Engineer,1205,,,not started
2,3,Computer Vision - AI Software Engineer,1299,,,not started
3,4,Senior Data Engineer,1842,,,not started
4,5,Business Application Developer,1456,,,not started
5,6,Fullstack Python/Django/React Developer,1654,,,not started
6,7,Big Data Engineer,2531,,,not started
7,8,Senior Python Backend Engineer - Relocate,1297,,,not started
8,9,Python Senior,1556,,,not started
9,10,Python Software Engineer,1228,,,not started


Evaluation and frozen gold alignment passed.
A separate blank annotation template was created in runtime outputs.


## Deterministic Pilot Design

Annotation orders 1 through 10 formed the initial pilot.

The pilot did not create a separate evaluation set. All 10 pilot documents
remained part of the final 100-document held-out corpus. The purpose was to
confirm that the annotation protocol, template, label inventory, and saving
workflow were usable before completing the remaining documents.

The table below is a read-only record of the fixed pilot-document identities.
It does not expose the original editable Drive working file.

In [16]:
pilotAnnotationOrders = list(
    range(
        1,
        11
    )
)


pilotDocuments = evaluationData.loc[
    evaluationData[
        "Annotation order"
    ].isin(
        pilotAnnotationOrders
    ),
    [
        "Annotation order",
        "id",
        "Position",
        "Company Name",
        "Primary Keyword",
        "charc_len"
    ]
].sort_values(
    "Annotation order"
).reset_index(
    drop=True
)


assert pilotDocuments[
    "Annotation order"
].tolist() == pilotAnnotationOrders, (
    "The deterministic pilot must contain annotation orders 1 through 10."
)


assert len(pilotDocuments) == 10, (
    "The deterministic pilot must contain 10 documents."
)


display(
    pilotDocuments
)


pilotRoleDistribution = pilotDocuments[
    "Primary Keyword"
].value_counts().rename_axis(
    "Role family"
).reset_index(
    name="Pilot documents"
)


display(
    pilotRoleDistribution
)


print(
    "Deterministic annotation pilot verified: "
    "orders 1 through 10."
)

,Annotation order,id,Position,Company Name,Primary Keyword,charc_len
0,1,0031e89e-0730-524b-bca2-7d7e5704dd3d,"Backend developer (Python, Golang or C#) Short...",Innovecs,Python,3733
1,2,04542ec5-2f52-555e-9453-5c39a1c41e2b,Data Engineer,MintyMint,Data Science,1205
2,3,0494e9e9-452b-552d-93c6-f484ede863e8,Computer Vision - AI Software Engineer,AllStars-IT,Data Science,1299
3,4,098ea499-fac1-5e9d-97a6-9c46bd7eb9b0,Senior Data Engineer,"Dynamo Development, Inc.",Python,1842
4,5,0aa39ab0-11f4-5efd-bf54-02d936cf594c,Business Application Developer,Svitla Systems,Data Engineer,1456
5,6,0aecc0af-b984-5cd8-945d-6a6553671341,Fullstack Python/Django/React Developer,K&C,Python,1654
6,7,0dae9522-7a5d-5888-8821-3ca2b3f03b23,Big Data Engineer,Intersog,Data Engineer,2531
7,8,0e74af31-07dc-5621-9f7c-5c49d3f4834a,Senior Python Backend Engineer - Relocate,Synergy Sports,Python,1297
8,9,1089b639-8242-5410-a887-16de33980af7,Python Senior,EPIC Conjoint,Python,1556
9,10,12e84961-cd2e-5199-8c45-5fb970568902,Python Software Engineer,reface.ai,Python,1228


,Role family,Pilot documents
0,Python,6
1,Data Science,2
2,Data Engineer,2


Deterministic annotation pilot verified: orders 1 through 10.


## Final Gold-Annotation Validation

The final gold file is validated structurally before it is used for system
evaluation.

The checks confirm that:

- all 100 annotation rows are complete;
- no row remains marked `needs review` or `not started`;
- every gold label belongs to the frozen 20-label inventory;
- no duplicate canonical labels occur within one document;
- no label appears in both `Gold skills` and `Uncertain`;
- the final gold rows remain aligned with the fixed held-out evaluation set.

These checks do not run System A, System B, aliases, regex matches, or
evaluation metrics.

In [17]:
def parseLabels(value):

    if pd.isna(value):
        return []

    cleanedValue = str(
        value
    ).strip()

    if cleanedValue in {
        "",
        "none"
    }:
        return []

    return [
        label.strip()
        for label in cleanedValue.split(
            ";"
        )
        if label.strip()
    ]


allowedCanonicalLabels = set(
    canonicalLabels
)


validationRows = []


for _, row in goldAnnotations.iterrows():

    goldLabels = parseLabels(
        row[
            "Gold skills"
        ]
    )

    uncertainLabels = parseLabels(
        row[
            "Uncertain"
        ]
    )

    annotationStatus = str(
        row[
            "Annotation status"
        ]
    ).strip()

    rowIssues = []

    if annotationStatus != "complete":

        rowIssues.append(
            "Annotation status is not complete: "
            f"{annotationStatus}"
        )

    if not str(
        row[
            "Gold skills"
        ]
    ).strip():

        rowIssues.append(
            "Gold skills field is blank."
        )

    if len(goldLabels) != len(
        set(
            goldLabels
        )
    ):

        rowIssues.append(
            "Duplicate label in Gold skills."
        )

    invalidGoldLabels = sorted(
        set(
            goldLabels
        )
        - allowedCanonicalLabels
    )

    if invalidGoldLabels:

        rowIssues.append(
            "Invalid Gold skills label(s): "
            + "; ".join(
                invalidGoldLabels
            )
        )

    invalidUncertainLabels = sorted(
        set(
            uncertainLabels
        )
        - allowedCanonicalLabels
    )

    if invalidUncertainLabels:

        rowIssues.append(
            "Invalid Uncertain label(s): "
            + "; ".join(
                invalidUncertainLabels
            )
        )

    overlappingLabels = sorted(
        set(
            goldLabels
        )
        & set(
            uncertainLabels
        )
    )

    if overlappingLabels:

        rowIssues.append(
            "Label appears in both Gold skills and Uncertain: "
            + "; ".join(
                overlappingLabels
            )
        )

    if uncertainLabels:

        rowIssues.append(
            "Unresolved uncertain label(s): "
            + "; ".join(
                uncertainLabels
            )
        )

    validationRows.append(
        {
            "Annotation order": row[
                "Annotation order"
            ],
            "Annotation status": annotationStatus,
            "Gold skills": row[
                "Gold skills"
            ],
            "Uncertain": row[
                "Uncertain"
            ],
            "Valid": len(
                rowIssues
            ) == 0,
            "Issues": "; ".join(
                rowIssues
            )
        }
    )


finalAnnotationValidation = pd.DataFrame(
    validationRows
)


invalidAnnotationRows = finalAnnotationValidation.loc[
    ~finalAnnotationValidation[
        "Valid"
    ]
].copy()


assert len(invalidAnnotationRows) == 0, (
    "The frozen final gold file has structural validation issues."
)


assert goldAnnotations[
    "Annotation status"
].eq(
    "complete"
).all(), (
    "Every released gold annotation must be complete."
)


finalAnnotationValidation.to_csv(
    runtimeFolder
    / "FinalAnnotationValidation.csv",
    index=False
)


validationSummary = pd.DataFrame(
    {
        "Measure": [
            "Final gold-annotation rows",
            "Complete rows",
            "Rows needing review",
            "Rows not started",
            "Structural validation issues"
        ],
        "Value": [
            len(goldAnnotations),
            int(
                goldAnnotations[
                    "Annotation status"
                ].eq(
                    "complete"
                ).sum()
            ),
            int(
                goldAnnotations[
                    "Annotation status"
                ].eq(
                    "needs review"
                ).sum()
            ),
            int(
                goldAnnotations[
                    "Annotation status"
                ].eq(
                    "not started"
                ).sum()
            ),
            len(invalidAnnotationRows)
        ]
    }
)


display(
    validationSummary
)


print(
    "Final gold-annotation structural validation passed."
)

,Measure,Value
0,Final gold-annotation rows,100
1,Complete rows,100
2,Rows needing review,0
3,Rows not started,0
4,Structural validation issues,0


Final gold-annotation structural validation passed.


## Gold-Annotation Summary

The following summaries describe the released final gold dataset. They are
descriptive properties of the frozen reference labels, not system-performance
results.

The label-frequency table reports the number of held-out documents containing
each canonical label. Sparse labels remain part of the frozen inventory, but
their individual performance estimates in Notebook 4 should be interpreted
with appropriate caution.

In [18]:
goldAnnotations[
    "Gold label set"
] = goldAnnotations[
    "Gold skills"
].apply(
    lambda value: set(
        parseLabels(
            value
        )
    )
)


goldAnnotations[
    "Uncertain label set"
] = goldAnnotations[
    "Uncertain"
].apply(
    lambda value: set(
        parseLabels(
            value
        )
    )
)


numberDocumentsWithLabels = int(
    goldAnnotations[
        "Gold label set"
    ].apply(
        len
    ).gt(
        0
    ).sum()
)


numberDocumentsWithNone = int(
    goldAnnotations[
        "Gold label set"
    ].apply(
        len
    ).eq(
        0
    ).sum()
)


totalGoldLabelInstances = int(
    goldAnnotations[
        "Gold label set"
    ].apply(
        len
    ).sum()
)


numberDocumentsWithUncertainty = int(
    goldAnnotations[
        "Uncertain label set"
    ].apply(
        len
    ).gt(
        0
    ).sum()
)


annotationSummary = pd.DataFrame(
    {
        "Measure": [
            "Held-out evaluation documents",
            "Completed gold annotations",
            "Documents with one or more gold labels",
            "Documents labelled none",
            "Gold-label instances",
            "Documents with unresolved uncertainty"
        ],
        "Value": [
            len(goldAnnotations),
            int(
                goldAnnotations[
                    "Annotation status"
                ].eq(
                    "complete"
                ).sum()
            ),
            numberDocumentsWithLabels,
            numberDocumentsWithNone,
            totalGoldLabelInstances,
            numberDocumentsWithUncertainty
        ]
    }
)


display(
    annotationSummary
)


goldLabelFrequency = pd.DataFrame(
    {
        "Canonical label": canonicalLabels
    }
)


goldLabelFrequency[
    "Gold-document count"
] = goldLabelFrequency[
    "Canonical label"
].apply(
    lambda label: int(
        goldAnnotations[
            "Gold label set"
        ].apply(
            lambda labelSet: label in labelSet
        ).sum()
    )
)


goldLabelFrequency = goldLabelFrequency.sort_values(
    [
        "Gold-document count",
        "Canonical label"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(
    drop=True
)


display(
    goldLabelFrequency
)


labelsPerDocument = goldAnnotations[
    "Gold label set"
].apply(
    len
)


labelsPerDocumentDistribution = labelsPerDocument.value_counts(
    sort=False
).rename_axis(
    "Gold labels in document"
).reset_index(
    name="Documents"
).sort_values(
    "Gold labels in document"
).reset_index(
    drop=True
)


display(
    labelsPerDocumentDistribution
)


goldLabelFrequency.to_csv(
    runtimeFolder
    / "GoldLabelFrequency.csv",
    index=False
)


labelsPerDocumentDistribution.to_csv(
    runtimeFolder
    / "LabelsPerDocumentDistribution.csv",
    index=False
)

,Measure,Value
0,Held-out evaluation documents,100
1,Completed gold annotations,100
2,Documents with one or more gold labels,95
3,Documents labelled none,5
4,Gold-label instances,379
5,Documents with unresolved uncertainty,0


,Canonical label,Gold-document count
0,python,83
1,sql,44
2,amazon web services,33
3,machine learning,33
4,postgresql,28
5,docker,20
6,google cloud platform,20
7,git,19
8,azure,16
9,mongodb,14


,Gold labels in document,Documents
0,0,5
1,1,8
2,2,16
3,3,17
4,4,20
5,5,12
6,6,12
7,7,5
8,8,4
9,9,1


## Read-Only Annotation Examples

The examples below show how completed frozen annotations are represented. They
are displayed for audit and methodological transparency only.

No editing controls are provided in the public reproduction workflow. The
official final gold file remains unchanged.

In [19]:
exampleAnnotationOrders = [
    1,
    10,
    50,
    100
]


annotationExamples = goldAnnotations.loc[
    goldAnnotations[
        "Annotation order"
    ].isin(
        exampleAnnotationOrders
    ),
    [
        "Annotation order",
        "Position",
        "Gold skills",
        "Uncertain",
        "Annotation status",
        "Long Description"
    ]
].copy()


annotationExamples[
    "Description preview"
] = annotationExamples[
    "Long Description"
].apply(
    lambda text: (
        str(text)[:900]
        + (
            "..."
            if len(
                str(text)
            ) > 900
            else ""
        )
    )
)


annotationExamples = annotationExamples[
    [
        "Annotation order",
        "Position",
        "Gold skills",
        "Uncertain",
        "Annotation status",
        "Description preview"
    ]
].sort_values(
    "Annotation order"
).reset_index(
    drop=True
)


display(
    annotationExamples
)

,Annotation order,Position,Gold skills,Uncertain,Annotation status,Description preview
0,1,"Backend developer (Python, Golang or C#) Short...",python,NaN,complete,We need bright talented engineers to help us d...
1,10,Python Software Engineer,google cloud platform; mongodb; postgresql; py...,NaN,complete,● 3+ years of software engineering experience ...
2,50,"Full Stack Developer (Python, React+GIS skills)",azure; docker; machine learning; mongodb; post...,NaN,complete,"As a Full Stack Developer, you will play a cru..."
3,100,Machine Learning Engineer,apache spark; machine learning; natural langua...,NaN,complete,Requirements:\r\n- No less than 2 years in ML ...


## Optional Practice Template and Reproducibility Note

A blank template is created in the runtime output folder for optional
independent practice annotation.

Any labels entered into a user-created template represent a separate annotation
exercise. They must not replace, overwrite, or be confused with the frozen
official gold annotations used to report the project’s System A versus System B
results.

In [20]:
practiceTemplatePath = (
    runtimeFolder
    / "UserPracticeAnnotationTemplate.csv"
)


goldAnnotationTemplate.to_csv(
    practiceTemplatePath,
    index=False
)


print(
    "Optional blank practice template created."
)


print(
    f"Template rows: {len(goldAnnotationTemplate)}"
)


print(
    "Official frozen gold file was not modified."
)


print(
    "Notebook 3 completed: annotation protocol, "
    "template construction, pilot design, final validation, "
    "dataset summary, and read-only examples are available."
)

Optional blank practice template created.
Template rows: 100
Official frozen gold file was not modified.
Notebook 3 completed: annotation protocol, template construction, pilot design, final validation, dataset summary, and read-only examples are available.


## Conclusion

The 100-document held-out corpus was annotated independently using a fixed
20-label canonical inventory and a full-document explicit-evidence protocol.

The released gold file passes structural validation:

- 100 annotation rows are complete.
- No row remains unresolved.
- All labels belong to the frozen canonical inventory.
- No duplicate or overlapping gold and uncertainty labels remain.
- The gold file remains exactly aligned with the fixed held-out evaluation
  documents.

The frozen final gold annotations are used as a read-only input in Notebook 4, which then compares System A and System B predictions against these labels; it
does not modify the annotation file, lexicon, or evaluation documents.

## Limitation

The released gold annotations were produced by one annotator using a fixed
protocol and closed 20-label inventory. They provide the reference labels for
this project’s held-out comparison, but they do not establish inter-annotator
agreement or universal coverage of all possible technical skills.